# GitHub Portfolio Agent

A multi-agent system that connects to your **real GitHub profile**, fetches your repos, and gives you an actionable portfolio audit.

**How it works:**
1. Enter your GitHub username — it fetches your real repos via GitHub API
2. Optionally enter your portfolio URL — it scrapes and analyzes it
3. Four specialized agents audit everything and give you a roadmap

**Agents:**
- **Repo Analyzer** — scores your real projects, finds strengths and gaps
- **README Writer** — drafts professional READMEs for your weakest repos
- **Portfolio Strategist** — actionable improvement plan with priorities
- **Recruiter Simulator** — honest hire/pass verdict with interview questions

**Week 8 Community Contribution by Mirack**

In [ ]:
import os
import json
import re
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

In [ ]:
client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
)

MODEL = "gpt-4.1-nano"

## Step 1: Enter Your Info

In [ ]:
# === You can customise this ===
GITHUB_USERNAME = "miracle73"
PORTFOLIO_URL = "https://mirack.site/"  
TARGET_ROLE = "AI/ML Engineer"

## Step 2: Fetch Real GitHub Data

In [ ]:
def fetch_github_profile(username):
    """Fetch real profile and repos from GitHub API (no auth needed for public repos)."""
    profile_resp = requests.get(f"https://api.github.com/users/{username}")
    if profile_resp.status_code != 200:
        raise Exception(f"Could not fetch profile for {username}: {profile_resp.status_code}")
    profile = profile_resp.json()
    
    repos_resp = requests.get(
        f"https://api.github.com/users/{username}/repos",
        params={"sort": "updated", "per_page": 30, "type": "owner"}
    )
    repos_raw = repos_resp.json()
    
    repos = []
    for r in repos_raw:
        if r.get("fork"):
            continue
        repos.append({
            "name": r["name"],
            "description": r.get("description") or "No description",
            "language": r.get("language") or "Unknown",
            "stars": r["stargazers_count"],
            "forks": r["forks_count"],
            "url": r["html_url"],
            "topics": r.get("topics", []),
            "updated": r["updated_at"][:10],
            "size_kb": r["size"]
        })
    
    return {
        "username": username,
        "name": profile.get("name") or username,
        "bio": profile.get("bio") or "No bio set",
        "public_repos": profile["public_repos"],
        "followers": profile["followers"],
        "profile_url": profile["html_url"],
        "repos": repos
    }

In [ ]:
print(f"Fetching GitHub data for: {GITHUB_USERNAME}...\n")
github_data = fetch_github_profile(GITHUB_USERNAME)

print(f"Name: {github_data['name']}")
print(f"Bio: {github_data['bio']}")
print(f"Public repos: {github_data['public_repos']} | Followers: {github_data['followers']}")
print(f"\nRepos found (excluding forks): {len(github_data['repos'])}")
print(f"\n{'Repo':<30} {'Language':<12} {'Stars':<6} {'Updated'}")
print("-" * 65)
for r in github_data['repos']:
    print(f"{r['name'][:29]:<30} {r['language']:<12} {r['stars']:<6} {r['updated']}")

## Step 3: Scrape Portfolio Website (Optional)

In [ ]:
def scrape_portfolio(url):
    """Scrape a portfolio website and extract key content."""
    if not url:
        return None
    try:
        resp = requests.get(url, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        return {"url": url, "content": text[:3000]}
    except Exception as e:
        print(f"Could not scrape portfolio: {e}")
        return None

portfolio_data = scrape_portfolio(PORTFOLIO_URL)
if portfolio_data:
    print(f"Portfolio scraped: {portfolio_data['url']} ({len(portfolio_data['content'])} chars)")
else:
    print("No portfolio URL provided — agents will focus on GitHub repos only.")

## Agent 1: Repo Analyzer

In [ ]:
def analyze_repos(github_data, target_role):
    repos_summary = json.dumps(github_data['repos'][:15], indent=2)
    portfolio_note = ""
    if portfolio_data:
        portfolio_note = f"\n\nPortfolio website content:\n{portfolio_data['content'][:1500]}"
    
    prompt = f"""Analyze this developer's REAL GitHub profile for: {target_role}

Profile: {github_data['name']} | Bio: {github_data['bio']} | {github_data['public_repos']} repos | {github_data['followers']} followers

Repos:
{repos_summary}
{portfolio_note}

Respond ONLY with JSON:
{{
    "languages_used": ["all languages across repos"],
    "strongest_areas": ["top 3 demonstrated skills"],
    "gaps_for_role": ["what's missing for {target_role}"],
    "best_repos": ["top 3 repo names"],
    "repos_needing_work": [{{"repo": "name", "issue": "what's wrong"}}],
    "profile_issues": ["problems with bio, missing descriptions, etc"],
    "portfolio_score": 1-10,
    "verdict": "one sentence summary"
}}"""
    
    print("\n[Repo Analyzer] Examining your real GitHub data...\n")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior engineering hiring manager. Analyze this REAL GitHub profile honestly."},
            {"role": "user", "content": prompt}
        ]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    result = json.loads(raw)
    
    score = result.get('portfolio_score') or result.get('score') or result.get('overall_score', 'N/A')
    verdict = result.get('verdict') or result.get('summary') or result.get('one_line_verdict', 'N/A')
    strengths = result.get('strongest_areas') or result.get('strengths') or result.get('strong_areas', [])
    gaps = result.get('gaps_for_role') or result.get('gaps') or result.get('weaknesses', [])
    best = result.get('best_repos') or result.get('top_repos', [])
    
    print(f"Portfolio Score: {score}/10")
    print(f"Verdict: {verdict}")
    print(f"Strongest: {', '.join(strengths) if isinstance(strengths, list) else strengths}")
    print(f"Gaps: {', '.join(gaps) if isinstance(gaps, list) else gaps}")
    print(f"Best repos: {', '.join(best) if isinstance(best, list) else best}")
    return result

In [ ]:
analysis = analyze_repos(github_data, TARGET_ROLE)

## Agent 2: README Writer

In [ ]:
def write_readme(github_data, analysis):
    weak = analysis.get('repos_needing_work', [])
    if weak:
        target_name = weak[0]['repo'] if isinstance(weak[0], dict) else weak[0]
    else:
        target_name = github_data['repos'][0]['name']
    
    target_repo = next((r for r in github_data['repos'] if r['name'] == target_name), github_data['repos'][0])
    
    prompt = f"""Write a professional GitHub README for this REAL project.
Clean and concise. Include: what it does, tech stack, how to run, what was learned.

Repo: {target_repo['name']}
Description: {target_repo['description']}
Language: {target_repo['language']}
Topics: {', '.join(target_repo.get('topics', []))}

Write in Markdown."""
    
    print(f"\n[README Writer] Drafting README for: {target_repo['name']}\n")
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You write clean GitHub READMEs. No fluff. A developer should understand the project in 30 seconds."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    print()
    return result

In [ ]:
readme = write_readme(github_data, analysis)

## Agent 3: Portfolio Strategist

In [ ]:
def build_strategy(github_data, analysis, target_role):
    prompt = f"""Create a concrete improvement plan for this REAL developer.

Target role: {target_role}
Profile: {github_data['name']} | {github_data['public_repos']} repos | {github_data['followers']} followers
Analysis: {json.dumps(analysis)}

Respond ONLY with JSON:
{{
    "quick_wins": [{{"action": "what to do", "impact": "high/medium/low", "time": "estimate"}}],
    "new_projects": [{{"name": "idea", "why": "fills what gap", "tech": ["stack"], "complexity": "weekend/week/month"}}],
    "repos_to_fix": [{{"repo": "name", "fixes": ["changes"]}}],
    "30_day_plan": ["week 1: ...", "week 2: ...", "week 3: ...", "week 4: ..."]
}}"""
    
    print("\n[Portfolio Strategist] Building your roadmap...\n")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a career coach for AI/ML engineers. Specific, actionable advice based on REAL repos."},
            {"role": "user", "content": prompt}
        ]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    result = json.loads(raw)
    
    print("QUICK WINS:")
    for qw in result['quick_wins']:
        print(f"  [{qw['impact'].upper():>6}] {qw['action']} (~{qw['time']})")
    
    print(f"\nNEW PROJECTS TO BUILD:")
    for p in result['new_projects']:
        print(f"  {p['name']} ({p['complexity']}) — {p['why']}")
    
    print(f"\n30-DAY PLAN:")
    for step in result['30_day_plan']:
        print(f"  {step}")
    return result

In [ ]:
strategy = build_strategy(github_data, analysis, TARGET_ROLE)

## Agent 4: Recruiter Simulator

In [ ]:
def recruiter_review(github_data, analysis, target_role):
    portfolio_section = ""
    if portfolio_data:
        portfolio_section = f"\nPortfolio site:\n{portfolio_data['content'][:1000]}"
    
    prompt = f"""You are a tech recruiter screening for: {target_role}

Candidate's GitHub:
- Name: {github_data['name']}
- Bio: {github_data['bio']}
- {github_data['public_repos']} repos | {github_data['followers']} followers
- Top repos: {json.dumps(github_data['repos'][:10], indent=2)}
- Analysis: {json.dumps(analysis)}
{portfolio_section}

60-second assessment:
1. First impression (10-second scan)
2. What impresses you
3. Red flags
4. Three interview questions based on their actual projects
5. Decision: HIRE / PHONE SCREEN / PASS — and why"""
    
    print("\n[Recruiter Simulator] Reviewing your profile...\n")
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a brutally honest senior recruiter at a top AI company. Be specific to this candidate's actual repos."},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="")
            result += content
    print()
    return result

In [ ]:
review = recruiter_review(github_data, analysis, TARGET_ROLE)

## Full Audit — Run on Any GitHub User

In [ ]:
def audit_portfolio(username, target_role, portfolio_url=""):
    """Complete portfolio audit — works on any public GitHub user."""
    print(f"{'#'*60}")
    print(f"PORTFOLIO AUDIT: github.com/{username}")
    print(f"Target Role: {target_role}")
    print(f"{'#'*60}\n")
    
    global portfolio_data
    data = fetch_github_profile(username)
    portfolio_data = scrape_portfolio(portfolio_url) if portfolio_url else None
    
    a = analyze_repos(data, target_role)
    r = write_readme(data, a)
    s = build_strategy(data, a, target_role)
    rev = recruiter_review(data, a, target_role)
    
    print(f"\n{'#'*60}")
    print("AUDIT COMPLETE")
    print(f"{'#'*60}")
    return {"analysis": a, "readme": r, "strategy": s, "review": rev}

In [ ]:
# You can try it on your public GitHub!
audit_portfolio("miracle73", "AI/ML Engineer", "https://www.mirack.site/")